# Pipeline de Entrenamiento del Modelo de Predicción de Suscripción

## Introducción

Este notebook tiene como propósito cargar las características previamente procesadas del dataset de marketing bancario, entrenar un modelo de clasificación de red neuronal profunda (DNN) para predecir la suscripción de clientes, evaluar su rendimiento y registrar el modelo, sus parámetros y métricas utilizando MLflow.

## 1. Importar Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import mlflow
import mlflow.tensorflow

print(f"TensorFlow Version: {tf.__version__}")
print(f"MLflow Version: {mlflow.__version__}")

## 2. Configuración de MLflow

Se define un nombre para el experimento en MLflow. Esto ayuda a organizar las ejecuciones.

In [ ]:
mlflow_experiment_name = "Bank_Subscription_Prediction_DNN_v1"
mlflow.set_experiment(mlflow_experiment_name)

print(f"MLflow Experiment set to: '{mlflow_experiment_name}'")

## 3. Cargar Datos Procesados

Cargamos el archivo `processed_bank_data.parquet` que fue generado por el `feature_pipeline.ipynb`. Se asume que este archivo se encuentra en un directorio relativo `../feature_pipeline/data/`.

In [ ]:
# Ajustar la ruta si es necesario. 
# Si feature_pipeline.ipynb guardó en 'data/processed_bank_data.parquet' relativo a su propia ubicación:
processed_data_path = '../feature_pipeline/data/processed_bank_data.parquet'

if not os.path.exists(processed_data_path):
    print(f"Error: El archivo de datos procesados '{processed_data_path}' no se encontró.")
    print("Asegúrate de que el notebook 'feature_pipeline.ipynb' se haya ejecutado correctamente y la ruta sea la correcta.")
    # Alternativamente, podrías intentar una ruta local si el archivo fue copiado/movido:
    # processed_data_path = './data/processed_bank_data.parquet'
    # if not os.path.exists(processed_data_path):
    #     print(f"Error: Tampoco se encontró en './data/processed_bank_data.parquet'. Deteniendo ejecución.")
    #     df_processed = None
    df_processed = None
else:
    print(f"Cargando datos desde: {processed_data_path}")
    df_processed = pd.read_parquet(processed_data_path)
    print("Datos procesados cargados exitosamente.")
    display(df_processed.head())

# Separar características (X) y objetivo (y)
if df_processed is not None:
    if 'y' not in df_processed.columns:
        print("Error: La columna objetivo 'y' no se encuentra en el DataFrame cargado.")
        X = None
        y = None
    else:
        X = df_processed.drop('y', axis=1)
        y = df_processed['y']
        print("\nForma de las características (X):", X.shape)
        print("Forma del objetivo (y):", y.shape)
else:
    X = None
    y = None

## 4. División de Datos

Dividimos los datos en conjuntos de entrenamiento, validación y prueba. Usaremos una proporción de 70% para entrenamiento, 15% para validación y 15% para prueba.

In [ ]:
if X is not None and y is not None:
    # Primero, dividir en entrenamiento (70%) y temporal (30% para validación+prueba)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y # estratificar por 'y' es importante para clases desbalanceadas
    )
    
    # Luego, dividir el temporal en validación (50% de 30% = 15% total) y prueba (50% de 30% = 15% total)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    
    print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} ({len(X_train)/len(X)*100:.2f}%)")
    print(f"Tamaño del conjunto de validación: {X_val.shape[0]} ({len(X_val)/len(X)*100:.2f}%)")
    print(f"Tamaño del conjunto de prueba: {X_test.shape[0]} ({len(X_test)/len(X)*100:.2f}%)")
else:
    print("No se pueden dividir los datos porque X o y no están definidos (probablemente debido a un error en la carga de datos).")

## 5. Definición del Modelo (Red Neuronal Densa con Keras)

Se define una red neuronal secuencial simple con capas densas y regularización mediante Dropout.

In [ ]:
if X_train is not None:
    input_dim = X_train.shape[1]
    
    model = Sequential([
        Input(shape=(input_dim,)), # Capa de entrada explícita
        Dense(units=128, activation='relu'),
        Dropout(0.3), # Regularización para prevenir sobreajuste
        Dense(units=64, activation='relu'),
        Dropout(0.3),
        Dense(units=1, activation='sigmoid') # Capa de salida para clasificación binaria (probabilidad)
    ])
    
    print("Resumen del Modelo:")
    model.summary()
else:
    print("No se puede definir el modelo porque X_train no está definido.")
    model = None

## 6. Compilación del Modelo

Se compila el modelo especificando el optimizador, la función de pérdida y las métricas a monitorear.

In [ ]:
if model is not None:
    # Hiperparámetros para la compilación
    learning_rate = 0.001
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')] # AUC es una buena métrica para clases desbalanceadas
    )
    print(f"Modelo compilado con tasa de aprendizaje: {learning_rate}")
else:
    print("El modelo no está definido, no se puede compilar.")

## 7. Entrenamiento del Modelo con MLflow

Entrenamos el modelo dentro de una ejecución de MLflow para registrar automáticamente (o manualmente) parámetros, métricas y el modelo.

In [ ]:
if model is not None and X_train is not None and y_train is not None and X_val is not None and y_val is not None:
    # Hiperparámetros para el entrenamiento
    epochs = 100 # Número máximo de épocas
    batch_size = 32
    
    # Callback para Early Stopping
    early_stopping = EarlyStopping(
        monitor='val_loss', # Métrica a monitorear
        patience=10,        # Número de épocas sin mejora antes de detener
        restore_best_weights=True # Restaurar los pesos del modelo de la mejor época
    )
    
    # Iniciar una ejecución de MLflow
    with mlflow.start_run(run_name="DNN_Training_Run") as run:
        print(f"MLflow Run ID: {run.info.run_id}")
        mlflow.tensorflow.autolog() # Habilitar autologging para Keras/TensorFlow

        # Registrar hiperparámetros manualmente (autolog también captura algunos)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("optimizer", "Adam")
        mlflow.log_param("loss_function", "binary_crossentropy")
        mlflow.log_param("input_shape", input_dim)
        mlflow.log_param("model_architecture", model.to_json()) # Guardar arquitectura como JSON
        
        print("\nIniciando entrenamiento del modelo...")
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stopping],
            verbose=1 # 0 = silent, 1 = progress bar, 2 = one line per epoch
        )
        
        print("Entrenamiento completado.")
        
        # Autologging debería haber registrado el modelo y las métricas.
        # Si se requiere logging manual del modelo (ej. con firma):
        # from mlflow.models.signature import infer_signature
        # signature = infer_signature(X_train, model.predict(X_train))
        # mlflow.keras.log_model(model, "keras-dnn-model", signature=signature)
        
        print(f"MLflow Run {run.info.run_id} finalizado. Revisa la UI de MLflow.")
else:
    print("No se puede entrenar el modelo debido a que algunos componentes (modelo, datos) no están definidos.")

## 8. Evaluación del Modelo

Evaluamos el modelo entrenado sobre el conjunto de prueba para obtener una estimación de su rendimiento en datos no vistos. Estas métricas también se registran en MLflow (autologging debería manejarlas, pero podemos calcular y registrar métricas adicionales).

In [ ]:
if model is not None and X_test is not None and y_test is not None and 'run' in locals():
    print("\nEvaluando el modelo en el conjunto de prueba...")
    test_loss, test_accuracy, test_auc = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"Pérdida en el conjunto de prueba (Test Loss): {test_loss:.4f}")
    print(f"Exactitud en el conjunto de prueba (Test Accuracy): {test_accuracy:.4f}")
    print(f"AUC en el conjunto de prueba (Test AUC): {test_auc:.4f}")
    
    # Generar predicciones de probabilidad
    y_pred_proba = model.predict(X_test)
    
    # Generar predicciones binarias (umbral de 0.5)
    y_pred_binary = (y_pred_proba > 0.5).astype(int)
    
    # Calcular métricas adicionales de scikit-learn
    roc_auc_sklearn = roc_auc_score(y_test, y_pred_proba)
    precision_sklearn = precision_score(y_test, y_pred_binary)
    recall_sklearn = recall_score(y_test, y_pred_binary)
    f1_sklearn = f1_score(y_test, y_pred_binary)
    
    print(f"ROC AUC (scikit-learn): {roc_auc_sklearn:.4f}")
    print(f"Precisión (scikit-learn): {precision_sklearn:.4f}")
    print(f"Recall (scikit-learn): {recall_sklearn:.4f}")
    print(f"F1-Score (scikit-learn): {f1_sklearn:.4f}")
    
    # Registrar estas métricas adicionales en MLflow (si no fueron capturadas por autolog)
    # Es posible que autolog ya las capture si están en model.evaluate o history.
    # Para asegurarse, se pueden loguear explícitamente dentro del run.
    # Como el 'run' ya terminó, para loguear más cosas, necesitaríamos reabrirlo o iniciar uno nuevo.
    # Por simplicidad, asumimos que autolog es suficiente o que esto se haría dentro del 'with mlflow.start_run()'.
    # Si se quisiera loguear ahora, se podría hacer:
    with mlflow.start_run(run_id=run.info.run_id): # Reabrir el run para agregar más métricas
        mlflow.log_metric("test_loss_eval", test_loss)
        mlflow.log_metric("test_accuracy_eval", test_accuracy)
        mlflow.log_metric("test_auc_eval", test_auc) # Keras AUC
        mlflow.log_metric("test_roc_auc_sklearn", roc_auc_sklearn)
        mlflow.log_metric("test_precision_sklearn", precision_sklearn)
        mlflow.log_metric("test_recall_sklearn", recall_sklearn)
        mlflow.log_metric("test_f1_score_sklearn", f1_sklearn)
    print("\nMétricas de prueba adicionales registradas en MLflow.")
    
else:
    print("No se puede evaluar el modelo porque algunos componentes (modelo, datos de prueba, run de MLflow) no están definidos.")

## 9. Guardado del Modelo

El modelo entrenado, junto con sus métricas y parámetros, ha sido registrado por MLflow (específicamente por `mlflow.tensorflow.autolog()` o `mlflow.keras.log_model()`). 

En un entorno de producción, MLflow se configuraría con un **backend de seguimiento persistente** (como una base de datos PostgreSQL o un servidor MLflow dedicado) y un **registro de modelos** (Model Registry) para gestionar el ciclo de vida de los modelos (staging, producción, archivado).

## 10. Generación de `requirements.txt`

Ejecuta la siguiente celda para imprimir las versiones de las bibliotecas clave utilizadas en este notebook. Copia esta salida a un archivo `requirements.txt` para asegurar la reproducibilidad del entorno.

In [ ]:
print("# --- requirements.txt para training_pipeline.ipynb ---")
print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
import sklearn
print(f"scikit-learn=={sklearn.__version__}")
print(f"tensorflow=={tf.__version__}")
print(f"mlflow=={mlflow.__version__}")
print(f"pyarrow") # Necesario para pd.read_parquet